# Moscow Land Cover Analysis — Colab runner

Тонкий управляющий ноутбук. Вся логика живёт в пакете `moscow_landcover`.
Здесь только окружение Colab: Drive, установка пакета, аутентификация GEE и вызовы пайплайна.

**Порядок:** 1 → 2 → 3 → 4 → 5, затем быстрый smoke-тест (1 год), и только потом полный прогон.

In [ ]:
# ==== 1. Окружение Colab: Google Drive ====
from google.colab import drive  # noqa: F401
drive.mount('/content/drive')

In [ ]:
# ==== 2. Клонирование/обновление пакета и установка ====
import os
import sys

REPO_URL = 'https://github.com/maxlansky1/moscow-landcover-analysis.git'
REPO_DIR = '/content/moscow-landcover-analysis'

if os.path.isdir(REPO_DIR):
    !git -C "{REPO_DIR}" pull -q
else:
    !git clone -q "{REPO_URL}" "{REPO_DIR}"

# ВАЖНО: используем !pip, а не %pip — в %pip подстановка {REPO_DIR} не работает.
!pip install -q -e "{REPO_DIR}"

# Colab подхватывает .pth-хук editable-установки только при старте ядра,
# поэтому добавляем src/ в sys.path напрямую — импорт работает сразу.
SRC_DIR = os.path.join(REPO_DIR, 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
import importlib
importlib.invalidate_caches()

import moscow_landcover
print('moscow_landcover', moscow_landcover.__version__)
print('импорт из:', moscow_landcover.__file__)

In [ ]:
# ==== 3. Переменные окружения (не секреты в коде) ====
import os

# Корень дерева проекта на Google Drive (там же, где 01_Data/Boundaries).
os.environ['MLC_PROJECT_PATH'] = '/content/drive/MyDrive/Moscow_LandCover_Analysis'

# Идентификатор проекта Google Cloud / Earth Engine.
os.environ['MLC_EE_PROJECT'] = 'moscow-landcover-analysis'

In [ ]:
# ==== 4. Конфигурация, пути, логирование, проверка границ ====
from pathlib import Path

from moscow_landcover.config import load_config, paths_from_config
from moscow_landcover.logging import setup_logging

cfg = load_config()
paths = paths_from_config(cfg, create=True)
setup_logging(paths['LOGS'])

print('BASE       :', cfg.base_path)
print('GEE project:', cfg.project_id)

for name in ('moscow_historical.geojson', 'moscow_greater.geojson'):
    p = Path(paths['BOUNDARIES']) / name
    print(p, '✅' if p.exists() else '❌ ОТСУТСТВУЕТ — тест не пойдёт дальше')

In [ ]:
# ==== 5. Аутентификация и проверка Earth Engine ====
from moscow_landcover import gee

gee.authenticate(auth_mode='colab')   # откроется ссылка авторизации
gee.initialize()                      # проект берётся из MLC_EE_PROJECT
gee.check()                           # сервер + коллекция + тестовый снимок

## Smoke-тест: 1 территория × 1 год

Полный цикл на одном годе (композит → ESA-точки → RF → OA/Kappa → площади).
Сначала убеждаемся, что всё работает, и только потом запускаем все 8 итераций.

In [ ]:
from moscow_landcover.pipeline import run

smoke_results, smoke_failed = run(cfg, plan=[('Historical', 2026)])
print('failed:', smoke_failed)

## Полный прогон: 5 (Historical) + 3 (Greater)

Результаты → `03_Intermediate/results_by_year.json`, лог → `04_Logs/run_*.log`.

In [ ]:
results, failed = run(cfg)
print(f'прогонов: {sum(len(v) for v in results.values())}, провалов: {len(failed)}')

## Диагностика (F1/F2) — необязательно

In [ ]:
from moscow_landcover.diagnostics import availability, run_diagnostics

f1 = availability(cfg)
f2 = run_diagnostics(cfg, f1)

## Итоговая сводка (H2)

In [ ]:
from moscow_landcover.report import summary

summary(cfg, results)

## Карта и визуализация

In [ ]:
from moscow_landcover import boundaries as boundaries_module
from moscow_landcover.viz import (
    add_boundary_layer, add_classification_layer, add_legend, create_base_map,
)

year = cfg.analysis.historical_years[-1]
territory = 'Historical'
boundaries, areas = boundaries_module.load_boundaries(paths)
result = results[territory][year]

m = create_base_map(territory, cfg.map)
add_boundary_layer(m, boundaries[territory])
add_classification_layer(m, result['classified'], year, analysis=cfg.analysis,
                         map_cfg=cfg.map)
add_legend(m, 'classification', cfg.map)
m

## Экспорт классификаций — необязательно

In [ ]:
from moscow_landcover.export import export_classifications_to_drive

export_classifications_to_drive(
    results, boundaries,
    drive_folder='Moscow_LandCover_Classifications',
)